# Chapter 8 Companion Notebook: Market Risk Management

This notebook reproduces every worked numerical example from Chapter 8 of *AI in Finance* (`chapter8.tex`): parametric/historical/Monte Carlo VaR and Expected Shortfall, matrix-based portfolio VaR and Component VaR, GARCH-based and filtered-historical-simulation VaR, VaR backtesting (Kupiec's test, the Basel traffic-light framework, and ES backtesting), liquidity-adjusted VaR and the NSFR, delta-normal and delta-gamma VaR for options, stress testing, correlation stress testing, reverse stress testing and a VaR limit-breach example, and a head-to-head test of GARCH(1,1) versus gradient boosting for volatility forecasting on simulated asymmetric (leverage-effect) data.

## 1. Value at Risk: parametric and historical methods (Section 8.3)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

returns = np.array([0.0006, 0.0042, -0.0027, -0.0101, -0.0049, -0.0113, 0.0013, 0.0167,
                     -0.0053, -0.0068, 0.0065, 0.0049, 0.0019, -0.0106, 0.0002, 0.0089,
                     -0.0155, -0.0049, -0.0222, -0.0149])
V = 10_000_000

mu = returns.mean()
sigma = returns.std(ddof=1)
print(f"mean = {mu:.4%}, std = {sigma:.4%}")

z95, z99 = norm.ppf(0.95), norm.ppf(0.99)
VaR95_param = -(mu - z95 * sigma) * V
VaR99_param = -(mu - z99 * sigma) * V
print(f"Parametric VaR95: ${VaR95_param:,.0f}")
print(f"Parametric VaR99: ${VaR99_param:,.0f}")

sorted_returns = np.sort(returns)
idx95 = int(np.floor(0.05 * len(returns)))
VaR95_hist = -sorted_returns[idx95] * V
print(f"\nSorted returns: {sorted_returns}")
print(f"Historical VaR95 (2nd-worst return): ${VaR95_hist:,.0f}")

mean = -0.3200%, std = 0.9374%
Parametric VaR95: $186,195
Parametric VaR99: $250,081

Sorted returns: [-0.0222 -0.0155 -0.0149 -0.0113 -0.0106 -0.0101 -0.0068 -0.0053 -0.0049
 -0.0049 -0.0027  0.0002  0.0006  0.0013  0.0019  0.0042  0.0049  0.0065
  0.0089  0.0167]
Historical VaR95 (2nd-worst return): $155,000


## 2. Portfolio VaR with the variance-covariance matrix (Section 8.3.1)

In [2]:
cov_matrix = np.array([[0.000414, -0.000198], [-0.000198, 0.000118]])
w = np.array([0.5, 0.5])
port_var = w @ cov_matrix @ w
port_std = np.sqrt(port_var)
print(f"Portfolio std: {port_std:.4%}")

V2 = 2_000_000
VaR95_matrix = z95 * port_std * V2
VaR99_matrix = z99 * port_std * V2
print(f"Matrix-based VaR95: ${VaR95_matrix:,.0f}")
print(f"Matrix-based VaR99: ${VaR99_matrix:,.0f}")

Portfolio std: 0.5831%
Matrix-based VaR95: $19,182
Matrix-based VaR99: $27,130


## 2b. Component VaR: decomposing portfolio risk by position (Section 8.3.2)

In [3]:
Sigma_w = cov_matrix @ w
comp_var = w * Sigma_w
print(f"Sigma @ w: {Sigma_w}")
print(f"Component variance: {comp_var}, sum={comp_var.sum():.6f} (check vs port_var={port_var:.6f})")

frac = comp_var / port_var
comp_VaR = frac * VaR95_matrix
print(f"Fractional contribution: {frac}")
print(f"Component VaR95: AssetA=${comp_VaR[0]:,.0f}, AssetB=${comp_VaR[1]:,.0f}, sum=${comp_VaR.sum():,.0f}")

Sigma @ w: [ 1.08e-04 -4.00e-05]
Component variance: [ 5.4e-05 -2.0e-05], sum=0.000034 (check vs port_var=0.000034)
Fractional contribution: [ 1.58823529 -0.58823529]
Component VaR95: AssetA=$30,466, AssetB=$-11,284, sum=$19,182


## 3. Expected Shortfall (Section 8.4)

In [4]:
tail = sorted_returns[:idx95 + 1]
ES95_hist = -tail.mean() * V
print(f"Historical ES95 (average of two worst returns): ${ES95_hist:,.0f}")

ES95_param = (sigma * norm.pdf(z95) / 0.05 - mu) * V
print(f"Parametric ES95: ${ES95_param:,.0f}")

Historical ES95 (average of two worst returns): $188,500
Parametric ES95: $225,366


## 4. Monte Carlo simulation for risk measurement (Section 8.5)

In [5]:
rng = np.random.default_rng(42)
sims = rng.normal(mu, sigma, 100_000)
sim_losses = -sims * V

VaR95_mc = np.percentile(sim_losses, 95)
VaR99_mc = np.percentile(sim_losses, 99)
ES95_mc = sim_losses[sim_losses >= VaR95_mc].mean()

print(f"Monte Carlo VaR95: ${VaR95_mc:,.0f}")
print(f"Monte Carlo VaR99: ${VaR99_mc:,.0f}")
print(f"Monte Carlo ES95:  ${ES95_mc:,.0f}")

Monte Carlo VaR95: $187,460
Monte Carlo VaR99: $251,690
Monte Carlo ES95:  $226,378


## 4b. Volatility-adjusted (GARCH-based) VaR (Section 8.5.1)

In [6]:
# GARCH(1,1) forecast volatilities from Chapter 7's recursion (t=3 pre-shock, t=4 post-shock)
sigma_t3, sigma_t4 = 0.0195, 0.0208
V_garch = 10_000_000

VaR95_t3 = z95 * sigma_t3 * V_garch
VaR95_t4 = z95 * sigma_t4 * V_garch
print(f"VaR95 pre-shock (sigma={sigma_t3:.2%}): ${VaR95_t3:,.0f}")
print(f"VaR95 post-shock (sigma={sigma_t4:.2%}): ${VaR95_t4:,.0f}")
print(f"Relative increase: {(VaR95_t4-VaR95_t3)/VaR95_t3:.2%}")

VaR95 pre-shock (sigma=1.95%): $320,746
VaR95 post-shock (sigma=2.08%): $342,130
Relative increase: 6.67%


## 5. Backtesting Value at Risk models (Section 8.6)

In [7]:
r_star_95 = mu - z95 * sigma
r_star_99 = mu - z99 * sigma
exceptions_95 = (returns < r_star_95).sum()
exceptions_99 = (returns < r_star_99).sum()

print(f"95% VaR return threshold: {r_star_95:.4%}")
print(f"Observed exceptions at 95%: {exceptions_95} (expected {0.05*len(returns):.1f})")
print(f"99% VaR return threshold: {r_star_99:.4%}")
print(f"Observed exceptions at 99%: {exceptions_99} (expected {0.01*len(returns):.1f})")

95% VaR return threshold: -1.8619%
Observed exceptions at 95%: 1 (expected 1.0)
99% VaR return threshold: -2.5008%
Observed exceptions at 99%: 0 (expected 0.2)


## 5b. Kupiec's proportion-of-failures test (Section 8.6)

In [8]:
p, n, x = 0.05, 20, 1
num = (1-p)**(n-x) * p**x
xn = x/n
den = (1-xn)**(n-x) * xn**x
LR_POF = -2*np.log(num/den)
print(f"Observed rate x/n = {xn:.4f} (target p={p})")
print(f"Kupiec LR_POF statistic: {LR_POF:.4f} (chi-sq(1) 95% critical value = 3.841)")

Observed rate x/n = 0.0500 (target p=0.05)
Kupiec LR_POF statistic: -0.0000 (chi-sq(1) 95% critical value = 3.841)


## 5c. Backtesting Expected Shortfall (Section 8.6.1)

In [9]:
exception_loss = -sorted_returns[0] * V  # worst return, the single 95% exception
print(f"Realized loss on exception day: ${exception_loss:,.0f}")
print(f"Parametric ES95 forecast: ${ES95_param:,.0f}")
print(f"Ratio (realized/forecast): {exception_loss/ES95_param:.3f}")

Realized loss on exception day: $222,000
Parametric ES95 forecast: $225,366
Ratio (realized/forecast): 0.985


## 6. Liquidity-adjusted VaR and the NSFR (Section 8.7)

In [10]:
spread_bps = 30
liquidity_cost = 0.5 * (spread_bps / 10_000) * V2
LVaR95 = VaR95_matrix + liquidity_cost
print(f"Liquidity cost adjustment: ${liquidity_cost:,.0f}")
print(f"Liquidity-adjusted VaR95: ${LVaR95:,.0f}")

# Liquidity Coverage Ratio
hqla = 50_000_000
net_outflows = 40_000_000
lcr = hqla / net_outflows
print(f"\nLiquidity Coverage Ratio: {lcr:.0%}")

Liquidity cost adjustment: $3,000
Liquidity-adjusted VaR95: $22,182

Liquidity Coverage Ratio: 125%


## 6b. VaR for nonlinear portfolios: delta-normal and delta-gamma (Section 8.7.1)

In [11]:
S0, K, r_bs, sigma_bs, T_bs = 50.0, 50.0, 0.04, 0.30, 1.0
d1 = (np.log(S0/K) + (r_bs + sigma_bs**2/2)*T_bs) / (sigma_bs*np.sqrt(T_bs))
d2 = d1 - sigma_bs*np.sqrt(T_bs)
C0 = S0*norm.cdf(d1) - K*np.exp(-r_bs*T_bs)*norm.cdf(d2)
delta = norm.cdf(d1)
gamma = norm.pdf(d1) / (S0*sigma_bs*np.sqrt(T_bs))
print(f"C0={C0:.4f} delta={delta:.4f} gamma={gamma:.4f}")

daily_sigma = sigma_bs / np.sqrt(252)
n_contracts = 1000

delta_dollar_stdev = delta * S0 * daily_sigma * n_contracts
VaR95_dn = z95 * delta_dollar_stdev
VaR99_dn = z99 * delta_dollar_stdev
print(f"\nDelta-normal VaR95: ${VaR95_dn:.2f}, VaR99: ${VaR99_dn:.2f}")

# Full revaluation
S_up_95 = S0*(1+z95*daily_sigma)
S_up_99 = S0*(1+z99*daily_sigma)
def bs_call(S):
    d1_ = (np.log(S/K) + (r_bs + sigma_bs**2/2)*T_bs) / (sigma_bs*np.sqrt(T_bs))
    d2_ = d1_ - sigma_bs*np.sqrt(T_bs)
    return S*norm.cdf(d1_) - K*np.exp(-r_bs*T_bs)*norm.cdf(d2_)

C_up_95, C_up_99 = bs_call(S_up_95), bs_call(S_up_99)
loss_full_95 = (C_up_95-C0)*n_contracts
loss_full_99 = (C_up_99-C0)*n_contracts
print(f"Full-revaluation loss95: ${loss_full_95:.2f}, loss99: ${loss_full_99:.2f}")

# Delta-gamma
dS95, dS99 = S0*z95*daily_sigma, S0*z99*daily_sigma
dg95 = n_contracts*(delta*dS95 + 0.5*gamma*dS95**2)
dg99 = n_contracts*(delta*dS99 + 0.5*gamma*dS99**2)
print(f"Delta-gamma loss95: ${dg95:.2f}, loss99: ${dg99:.2f}")

C0=6.8766 delta=0.6115 gamma=0.0255

Delta-normal VaR95: $950.48, VaR99: $1344.28
Full-revaluation loss95: $980.71, loss99: $1404.20
Delta-gamma loss95: $981.34, loss99: $1406.01


## 7. Stress testing and correlation stress testing (Section 8.9)

In [12]:
stress_return = -0.07
stress_loss = -stress_return * V
print(f"Stressed loss (single-day -7% return): ${stress_loss:,.0f}")
print(f"Compare to parametric VaR99: ${VaR99_param:,.0f}")

# Correlation stress test
sigA, sigB = np.sqrt(cov_matrix[0,0]), np.sqrt(cov_matrix[1,1])
rho_calm = cov_matrix[0,1] / (sigA*sigB)
print(f"\nsigA={sigA:.4%} sigB={sigB:.4%} rho_calm={rho_calm:.4f}")

rho_stress = 0.5
cov_stress = rho_stress * sigA * sigB
Sigma_stress = np.array([[cov_matrix[0,0], cov_stress],[cov_stress, cov_matrix[1,1]]])
port_var_stress = w @ Sigma_stress @ w
port_std_stress = np.sqrt(port_var_stress)
print(f"Stressed portfolio std: {port_std_stress:.4%}")

VaR95_stress = z95 * port_std_stress * V2
print(f"VaR95 stress: ${VaR95_stress:,.0f}  (ratio to calm VaR95={VaR95_stress/VaR95_matrix:.2f}x)")

Stressed loss (single-day -7% return): $700,000
Compare to parametric VaR99: $250,081

sigA=2.0347% sigB=1.0863% rho_calm=-0.8958
Stressed portfolio std: 1.3721%
VaR95 stress: $45,137  (ratio to calm VaR95=2.35x)


## 7b. Reverse stress testing and a VaR limit breach (Section 8.9, Section 8.11)

Reverse stress test: what single-day return exhausts a $500,000 risk capital cushion? VaR limit example: utilization of a $300,000 approved limit, and a subsequent breach.

In [13]:
# Reverse stress test
cushion = 500_000
reverse_stress_return = cushion / V
print(f"Return that exhausts a ${cushion:,.0f} cushion: {reverse_stress_return:.2%}")
print(f"Historical -7% stress loss (${stress_loss:,.0f}) exceeds cushion by "
      f"${stress_loss - cushion:,.0f}")

# VaR limit utilization and breach
approved_limit = 300_000
utilization = VaR99_param / approved_limit
print(f"\nVaR99 limit utilization: {utilization:.1%}")

breached_var = 340_000
breach_amount = breached_var - approved_limit
print(f"Breach: ${breach_amount:,.0f} ({breach_amount/approved_limit:.1%} over limit)")

Return that exhausts a $500,000 cushion: 5.00%
Historical -7% stress loss ($700,000) exceeds cushion by $200,000

VaR99 limit utilization: 83.4%
Breach: $40,000 (13.3% over limit)


## 8. Machine learning for volatility forecasting: GARCH(1,1) vs. gradient boosting (Section 8.9.1)

Chapter 7 noted that a plain GARCH(1,1) model treats a positive and a negative shock of the same size as equally informative about future volatility, a well-documented shortcoming called the leverage effect. Here we test whether a gradient-boosted tree, given explicit access to the sign of each lagged return (information a symmetric GARCH(1,1) cannot use), can out-forecast GARCH(1,1) on a return series simulated from a *known* asymmetric (GJR-GARCH-style) process, where negative shocks raise variance more than equally-sized positive ones.

In [14]:
from arch import arch_model
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Simulate 750 days from a known asymmetric (GJR-GARCH-style) process:
# negative shocks get an extra variance kick (gamma) that positive shocks do not.
rng_vol = np.random.default_rng(7)
n_days_vol = 750
omega_true, alpha_true, beta_true, gamma_true = 0.000005, 0.05, 0.85, 0.12

sigma2_true = np.zeros(n_days_vol)
r_sim = np.zeros(n_days_vol)
sigma2_true[0] = omega_true / (1 - alpha_true - beta_true - gamma_true / 2)
r_sim[0] = rng_vol.normal(0, np.sqrt(sigma2_true[0]))
for t in range(1, n_days_vol):
    shock = r_sim[t - 1]
    leverage_term = gamma_true * shock**2 * (1.0 if shock < 0 else 0.0)
    sigma2_true[t] = omega_true + alpha_true * shock**2 + leverage_term + beta_true * sigma2_true[t - 1]
    r_sim[t] = rng_vol.normal(0, np.sqrt(sigma2_true[t]))

n_train_vol = 600
r_train, r_test = r_sim[:n_train_vol], r_sim[n_train_vol:]
sigma2_test_true = sigma2_true[n_train_vol:]
print(f"Simulated {n_days_vol} days ({n_train_vol} train, {n_days_vol - n_train_vol} test)")

Simulated 750 days (600 train, 150 test)


In [15]:
# Fit a plain, symmetric GARCH(1,1) via maximum likelihood -- structurally
# misspecified for this asymmetric process, since it cannot see the shock's sign.
am = arch_model(r_train * 100, mean="Zero", vol="GARCH", p=1, q=1, dist="normal")
res = am.fit(disp="off")
omega_hat = res.params["omega"] / 100**2
alpha_hat = res.params["alpha[1]"]
beta_hat = res.params["beta[1]"]
print(f"GARCH(1,1) fitted: omega={omega_hat:.3e}, alpha={alpha_hat:.4f}, beta={beta_hat:.4f}")

# One-step-ahead walk-forward forecasts over the test period
last_sigma2 = res.conditional_volatility[-1]**2 / 100**2
last_r = r_train[-1]
garch_forecasts = []
for t in range(len(r_test)):
    next_sigma2 = omega_hat + alpha_hat * last_r**2 + beta_hat * last_sigma2
    garch_forecasts.append(next_sigma2)
    last_sigma2 = next_sigma2
    last_r = r_test[t]
garch_forecasts = np.array(garch_forecasts)
garch_rmse = np.sqrt(np.mean((garch_forecasts - sigma2_test_true)**2))
print(f"GARCH(1,1) forecast-variance RMSE vs. true simulated variance: {garch_rmse:.3e}")

GARCH(1,1) fitted: omega=7.866e-06, alpha=0.1597, beta=0.7641
GARCH(1,1) forecast-variance RMSE vs. true simulated variance: 2.416e-05


In [16]:
# Gradient-boosted regressor with sign-aware features (lagged squared AND signed
# returns, so the model *can* in principle learn the leverage asymmetry GARCH(1,1)
# cannot represent), hyperparameters chosen by time-series-aware cross-validation.
sq_sim = r_sim**2
X_vol, y_vol = [], []
for t in range(2, n_train_vol):
    X_vol.append([sq_sim[t - 1], sq_sim[t - 2], r_sim[t - 1], r_sim[t - 2]])
    y_vol.append(sq_sim[t])
X_vol, y_vol = np.array(X_vol), np.array(y_vol)

param_grid = {"n_estimators": [10, 25, 50, 100], "max_depth": [1, 2, 3], "learning_rate": [0.01, 0.05, 0.1]}
tscv = TimeSeriesSplit(n_splits=5)
gs = GridSearchCV(GradientBoostingRegressor(random_state=0), param_grid, cv=tscv, scoring="neg_mean_squared_error")
gs.fit(X_vol, y_vol)
gbm_vol = gs.best_estimator_
print(f"Best GBM hyperparameters: {gs.best_params_}")

gbm_forecasts = np.array([
    gbm_vol.predict([[sq_sim[t - 1], sq_sim[t - 2], r_sim[t - 1], r_sim[t - 2]]])[0]
    for t in range(n_train_vol, n_days_vol)
])
gbm_rmse = np.sqrt(np.mean((gbm_forecasts - sigma2_test_true)**2))
print(f"Gradient boosting forecast-variance RMSE vs. true simulated variance: {gbm_rmse:.3e}")

naive_rmse = np.sqrt(np.mean((np.var(r_train) - sigma2_test_true)**2))
print(f"Naive (constant unconditional variance) RMSE vs. true simulated variance: {naive_rmse:.3e}")

print(f"\nGARCH(1,1) beats gradient boosting by {(1 - garch_rmse/gbm_rmse)*100:.1f}%")
print(f"Gradient boosting beats the naive benchmark by {(1 - gbm_rmse/naive_rmse)*100:.1f}%")

Best GBM hyperparameters: {'learning_rate': 0.05, 'max_depth': 1, 'n_estimators': 100}
Gradient boosting forecast-variance RMSE vs. true simulated variance: 3.830e-05
Naive (constant unconditional variance) RMSE vs. true simulated variance: 4.202e-05

GARCH(1,1) beats gradient boosting by 36.9%
Gradient boosting beats the naive benchmark by 8.8%


## Exercises (match Chapter 8, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [17]:
# Exercise 2: parametric 90% VaR
z90 = norm.ppf(0.90)
VaR90_param = -(mu - z90 * sigma) * V
print(f"Exercise 2 -- Parametric VaR90: ${VaR90_param:,.0f}")

# Exercise 8: Basel traffic-light classification, 7 exceptions in 250 days at 99%
exceptions_250 = 7
zone = "green" if exceptions_250 <= 4 else ("yellow" if exceptions_250 <= 9 else "red")
print(f"Exercise 8 -- {exceptions_250} exceptions in 250 days -> {zone} zone")

# Exercise 12: stressed correlation rho=0.8
rho_ex12 = 0.8
cov_ex12 = rho_ex12*sigA*sigB
Sigma_ex12 = np.array([[cov_matrix[0,0], cov_ex12],[cov_ex12, cov_matrix[1,1]]])
port_std_ex12 = np.sqrt(w @ Sigma_ex12 @ w)
VaR95_ex12 = z95*port_std_ex12*V2
print(f"Exercise 12 -- rho=0.8: portfolio std={port_std_ex12:.4%}, VaR95=${VaR95_ex12:,.0f}")

# Exercise 13: LCR
hqla_ex13, outflows_ex13 = 60_000_000, 55_000_000
print(f"Exercise 13 -- LCR: {hqla_ex13/outflows_ex13:.1%}")

# Exercise (limit utilization): tighter $275,000 limit
tighter_limit = 275_000
util_tighter = VaR99_param / tighter_limit
print(f"\nExercise (limit) -- utilization at $275,000 limit: {util_tighter:.1%}")
print(f"Would $290,000 breach this limit? {290_000 > tighter_limit}")

# Exercise (reverse stress test): $800,000 cushion
cushion_ex = 800_000
reverse_return_ex = cushion_ex / V
print(f"\nExercise (reverse stress) -- return exhausting $800,000 cushion: {reverse_return_ex:.2%}")
print(f"Does the -7% stress loss (${stress_loss:,.0f}) exceed this cushion? {stress_loss > cushion_ex}")

Exercise 2 -- Parametric VaR90: $152,137
Exercise 8 -- 7 exceptions in 250 days -> yellow zone
Exercise 12 -- rho=0.8: portfolio std=1.4880%, VaR95=$48,950
Exercise 13 -- LCR: 109.1%

Exercise (limit) -- utilization at $275,000 limit: 90.9%
Would $290,000 breach this limit? True

Exercise (reverse stress) -- return exhausting $800,000 cushion: 8.00%
Does the -7% stress loss ($700,000) exceed this cushion? False
